In [1]:
import pandas as pd

In [2]:
rides = pd.read_csv("/Users/kcarnold/Downloads/2011-capitalbikeshare-tripdata.zip")
rides.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1226767 entries, 0 to 1226766
Data columns (total 9 columns):
Duration                1226767 non-null int64
Start date              1226767 non-null object
End date                1226767 non-null object
Start station number    1226767 non-null int64
Start station           1226767 non-null object
End station number      1226767 non-null int64
End station             1226767 non-null object
Bike number             1226767 non-null object
Member type             1226767 non-null object
dtypes: int64(3), object(6)
memory usage: 84.2+ MB


In [7]:
rides.head()

,Duration,Start date,End date,Start station number,Start station,End station number,End station,Bike number,Member type,start
0,3548,2011-01-01 00:01:29,2011-01-01 01:00:37,31620,5th & F St NW,31620,5th & F St NW,W00247,Member,2011-01-01 00:01:29
1,346,2011-01-01 00:02:46,2011-01-01 00:08:32,31105,14th & Harvard St NW,31101,14th & V St NW,W00675,Casual,2011-01-01 00:02:46
2,562,2011-01-01 00:06:13,2011-01-01 00:15:36,31400,Georgia & New Hampshire Ave NW,31104,Adams Mill & Columbia Rd NW,W00357,Member,2011-01-01 00:06:13
3,434,2011-01-01 00:09:21,2011-01-01 00:16:36,31111,10th & U St NW,31503,Florida Ave & R St NW,W00970,Member,2011-01-01 00:09:21
4,233,2011-01-01 00:28:26,2011-01-01 00:32:19,31104,Adams Mill & Columbia Rd NW,31106,Calvert & Biltmore St NW,W00346,Casual,2011-01-01 00:28:26


In [8]:
del rides["Start station"]
del rides["End station"]

In [4]:
rides['Member type'].value_counts()

Member     979814
Casual     246949
Unknown         4
Name: Member type, dtype: int64

In [5]:
rides = rides[rides['Member type'] != "Unknown"]

In [6]:
rides['Member type'].value_counts()

Member    979814
Casual    246949
Name: Member type, dtype: int64

In [9]:
rides['start'] = pd.to_datetime(rides['Start date'])
rides['start'].iloc[0]

Timestamp('2011-01-01 00:01:29')

In [14]:
rides['start'].tail(10).dt.strftime("%Y-%m-%d")

1226757    2011-12-31
1226758    2011-12-31
1226759    2011-12-31
1226760    2011-12-31
1226761    2011-12-31
1226762    2011-12-31
1226763    2011-12-31
1226764    2011-12-31
1226765    2011-12-31
1226766    2011-12-31
Name: start, dtype: object

In [55]:
rides['date'] = rides['start'].dt.date#strftime("%Y-%m-%d")
rides['hour'] = rides['start'].dt.hour

In [56]:
grouped_as_index = rides.groupby(['date', 'hour', 'Member type']).size()
rides_by_hour = grouped_as_index.to_frame("rides").reset_index()
rides_by_hour

,date,hour,Member type,rides
0,2011-01-01,0,Casual,3
1,2011-01-01,0,Member,13
2,2011-01-01,1,Casual,8
3,2011-01-01,1,Member,30
4,2011-01-01,2,Casual,5
5,2011-01-01,2,Member,26
6,2011-01-01,3,Casual,3
7,2011-01-01,3,Member,9
8,2011-01-01,4,Member,1
9,2011-01-01,5,Member,1


In [57]:
from pandas.tseries.holiday import USFederalHolidayCalendar

In [58]:
import datetime

In [59]:
holidays = pd.DataFrame({
    'date': USFederalHolidayCalendar().holidays(datetime.date(2011,1,1), datetime.date(2015,12,31)).date,
    'is_holiday': True})
holidays.head()

,date,is_holiday
0,2011-01-17,True
1,2011-02-21,True
2,2011-05-30,True
3,2011-07-04,True
4,2011-09-05,True


In [63]:
# too few
rides_by_hour_with_holidays = pd.merge(
    rides_by_hour,
    holidays,
    left_on='date',
    right_on='date')
rides_by_hour_with_holidays.head()

,date,hour,Member type,rides,is_holiday
0,2011-01-17,0,Casual,1,True
1,2011-01-17,0,Member,16,True
2,2011-01-17,1,Casual,1,True
3,2011-01-17,1,Member,15,True
4,2011-01-17,2,Member,8,True


In [66]:
rides_by_hour_with_holidays = pd.merge(
    rides_by_hour,
    holidays,
    left_on='date',
    right_on='date',
    how='left',
    indicator=True)
rides_by_hour_with_holidays.head()

,date,hour,Member type,rides,is_holiday,_merge
0,2011-01-01,0,Casual,3,NaN,left_only
1,2011-01-01,0,Member,13,NaN,left_only
2,2011-01-01,1,Casual,8,NaN,left_only
3,2011-01-01,1,Member,30,NaN,left_only
4,2011-01-01,2,Casual,5,NaN,left_only


In [67]:
rides_by_hour_with_holidays['is_holiday'] = rides_by_hour_with_holidays['is_holiday'].fillna(False)

In [69]:
rides_by_hour_with_holidays.head()

,date,hour,Member type,rides,is_holiday,_merge
0,2011-01-01,0,Casual,3,False,left_only
1,2011-01-01,0,Member,13,False,left_only
2,2011-01-01,1,Casual,8,False,left_only
3,2011-01-01,1,Member,30,False,left_only
4,2011-01-01,2,Casual,5,False,left_only


https://www.ncei.noaa.gov/data/global-hourly/doc/isd-format-document.pdf
https://www.ncei.noaa.gov/data/global-hourly/doc/CSV_HELP.pdf

There's a "Find a Station" tool, but it's confusing how to use the results. https://www.ncdc.noaa.gov/data-access/land-based-station-data/station-metadata has a link to a [station list file](ftp://ftp.ncdc.noaa.gov/pub/data/noaa/isd-history.txt). Searching that, it looks like the code for Reagan Airport is 724050 13743. So the file is
https://www.ncei.noaa.gov/data/global-hourly/access/2011/72405013743.csv

In [70]:
weather = pd.read_csv("https://www.ncei.noaa.gov/data/global-hourly/access/2011/72405013743.csv")

//anaconda3/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3057: DtypeWarning: Columns (43,47,51,55) have mixed types. Specify dtype option on import or set low_memory=False.
  interactivity=interactivity, compiler=compiler, result=result)


In [75]:
weather['date'] = pd.to_datetime(weather['DATE'])

In [80]:
weather['date_hr'] = weather.date.dt.round(freq="H")

In [87]:
weather['TMP'].str.split(',', n=1, expand=True)

,0,1
0,+0056,1
1,+0056,5
2,+0050,5
3,+0039,5
4,+0039,1
5,+0022,5
6,+0033,5
7,+9999,9
8,+9999,9
9,+0017,5


In [90]:
weather[['temp', 'temp_validity']] = weather['TMP'].str.split(',', n=1, expand=True)

In [92]:
weather['temp'] = pd.to_numeric(weather['temp'])

In [95]:
weather[['temp','temp_validity']]

,temp,temp_validity
0,56,1
1,56,5
2,50,5
3,39,5
4,39,1
5,22,5
6,33,5
7,9999,9
8,9999,9
9,17,5


In [96]:
weather['temp_validity'].value_counts()

5    11266
1     2906
9      378
A        7
6        1
Name: temp_validity, dtype: int64